# MILK10k — Step 7: Training v2 (accuracy · stability · cost)
Upgraded version of notebook 02. **Same folds and same inner validation split** (split seed 42), so every downstream notebook works unchanged and v1 vs v2 is a clean ablation.

| Goal | Change vs v1 |
|---|---|
| Accuracy (melanoma) | Dermoscopy at **384 px** (ConvNeXtV2-tiny 22k-384 weights); clinical stays 224 px |
| Accuracy (colour cast) | **Shades-of-Gray colour constancy** (p = 6) on both image types |
| Stop early overfitting | Lower LR (5e-5), **layer-wise LR decay**, **stochastic depth 0.2**, label smoothing 0.1 |
| Stability | **EMA weights** (checkpoint = best EMA on validation); per-fold seeding; run **3 training seeds** (42, 43, 44) |
| Robust predictions | **8-view test-time augmentation** for dermoscopy (4 rotations × flip), 2-view for clinical |
| Cost | Images pre-resized + colour-corrected **once** into a cache (no large-JPEG decoding per epoch); channels-last; fp16; run time and GPU memory logged per model |

**Run once per training seed** (`TRAIN_SEED = 42`, then 43, then 44), each as **Save & Run All (Commit)**. Outputs go to `/kaggle/working/stage2v2_s{seed}/`. Settings: **GPU T4**, **Internet On**, attach **MILK10_DATASET**. Expected ~2–3 GPU hours per seed (check the log after fold 0).

In [ ]:
import os, time, copy, math, json, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from multiprocessing import Pool
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import train_test_split, StratifiedKFold

# ================= CONFIG =================
TRAIN_SEED = 42            # <- change to 43, then 44 (one commit each)
SPLIT_SEED, K_FOLDS = 42, 5   # never change: identical folds + inner validation split as v1
MOD_CFG = {
    "D": dict(size=384, backbone="facebook/convnextv2-tiny-22k-384", batch=16, accum=2, epochs=12, lr=5e-5),
    "C": dict(size=224, backbone="facebook/convnextv2-tiny-22k-224", batch=32, accum=1, epochs=12, lr=5e-5),
}
LLRD, DROP_PATH, EMA_DECAY, LABEL_SMOOTH, WD = 0.8, 0.2, 0.998, 0.1, 0.05
COLOR_CONSTANCY = True
MODS_TO_RUN, FOLDS_TO_RUN = ["D", "C"], [0, 1, 2, 3, 4]
OUT = f"/kaggle/working/stage2v2_s{TRAIN_SEED}"; os.makedirs(OUT, exist_ok=True)
CACHE = "/kaggle/working/cache"; os.makedirs(CACHE, exist_ok=True)

# ================= PATHS =================
def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None
SUPP_DIR = os.path.dirname(find_file("training_gt.csv")); assert SUPP_DIR, "attach MILK10_DATASET"
print("supplements:", SUPP_DIR, "| seed:", TRAIN_SEED, "| out:", OUT)

## Folds (rebuilt exactly as in notebook 02; verified against v1 counts)

In [ ]:
gt  = pd.read_csv(f"{SUPP_DIR}/training_gt.csv"); inp = pd.read_csv(f"{SUPP_DIR}/training_input.csv")
CL = [c for c in gt.columns if c != "lesion_id"]
gt["label"] = gt[CL].idxmax(1); gt["y"] = gt["label"].map({c: i for i, c in enumerate(CL)})
inp["mod"] = inp["image_type"].map({"clinical: close-up": "C", "dermoscopic": "D"})
MON = [c for c in inp.columns if c.startswith("MONET_")]
w = inp.pivot(index="lesion_id", columns="mod", values=["isic_id", "image_manipulation"] + MON)
w.columns = [f"{a}_{b}" for a, b in w.columns]
les = (inp[inp["mod"] == "D"][["lesion_id", "age_approx", "sex", "skin_tone_class", "site"]]
       .merge(w.reset_index(), on="lesion_id").merge(gt[["lesion_id", "label", "y"]], on="lesion_id"))
les["tone_group"] = pd.cut(les["skin_tone_class"], [-1, 2, 3, 5], labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str)
strata = les["label"] + "|" + les["tone_group"]
strata = np.where(strata.map(strata.value_counts()) >= K_FOLDS, strata, les["label"])
les["fold"] = -1
for f, (_, te) in enumerate(StratifiedKFold(K_FOLDS, shuffle=True, random_state=SPLIT_SEED).split(les, strata)):
    les.loc[les.index[te], "fold"] = f
les.to_csv(f"{OUT}/milk10k_lesions_folds.csv", index=False)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
assert CL == CLASSES and les["fold"].value_counts().sort_index().tolist() == [1048] * 5
v1 = find_file("milk10k_lesions_folds.csv", tops=("/kaggle/input",))
if v1:   # optional: if the v1 notebook output is attached, check the folds are identical
    same = (pd.read_csv(v1).set_index("lesion_id")["fold"].reindex(les["lesion_id"]).values == les["fold"].values).all()
    print("identical to v1 folds:", same); assert same
print(les.shape, "| lesions per fold:", les["fold"].value_counts().sort_index().tolist())

## Image cache: resize once (short side = 1.15 × training size) + Shades-of-Gray colour constancy

In [ ]:
IMG_DIR = os.path.dirname(find_file(les["isic_id_D"].iloc[0] + ".jpg")); print("images:", IMG_DIR)

def shades_of_gray(im, p=6):
    a = np.asarray(im).astype(np.float32)
    illum = np.power(np.mean(np.power(a, p), axis=(0, 1)), 1 / p)
    illum = illum / (np.sqrt((illum ** 2).sum()) + 1e-6)
    a = a * (1.0 / (illum * np.sqrt(3) + 1e-6))
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def cache_one(args):
    src, dst, short = args
    if os.path.exists(dst): return
    im = Image.open(src).convert("RGB"); w_, h_ = im.size; s = short / min(w_, h_)
    im = im.resize((max(short, round(w_ * s)), max(short, round(h_ * s))), Image.BICUBIC)
    if COLOR_CONSTANCY: im = shades_of_gray(im)
    im.save(dst, quality=95)

t0 = time.time()
for mod in MODS_TO_RUN:
    size = MOD_CFG[mod]["size"]; d = f"{CACHE}/{mod}_{size}{'_cc' if COLOR_CONSTANCY else ''}"; os.makedirs(d, exist_ok=True)
    jobs = [(f"{IMG_DIR}/{i}.jpg", f"{d}/{i}.jpg", int(size * 1.15)) for i in les[f"isic_id_{mod}"]]
    with Pool(4) as pool: pool.map(cache_one, jobs, chunksize=32)
    les[f"path_{mod}"] = [j[1] for j in jobs]
print(f"cache ready in {time.time() - t0:.0f}s")

## Metrics

In [ ]:
def evaluate(y, P):
    pred = P.argmax(1)
    out = {"bal_acc": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
           "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)])}
    try: out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

## Model, EMA, layer-wise LR decay, TTA

In [ ]:
import random, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from transformers import AutoModel
device = torch.device("cuda"); torch.backends.cudnn.benchmark = True
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def transforms(mod, train):
    s = MOD_CFG[mod]["size"]
    if not train: return T.Compose([T.Resize((s, s)), T.ToTensor(), T.Normalize(MEAN, STD)])
    if mod == "D":
        return T.Compose([T.RandomResizedCrop(s, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                          T.RandomRotation(180), T.ColorJitter(0.15, 0.15, 0.1, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])
    return T.Compose([T.RandomResizedCrop(s, scale=(0.7, 1.0)), T.RandomHorizontalFlip(), T.RandomRotation(20),
                      T.ColorJitter(0.15, 0.15, 0.1, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])

class LesionImages(Dataset):
    def __init__(self, df, mod, tf): self.p, self.y, self.tf = df[f"path_{mod}"].values, df["y"].values, tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

class Net(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone, drop_path_rate=DROP_PATH)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.config.hidden_sizes[-1], K))
    def embed(self, x): return self.backbone(pixel_values=x).pooler_output
    def forward(self, x): return self.head(self.embed(x))

def param_groups(net, lr):
    bb = net.backbone
    g = [{"params": list(bb.embeddings.parameters()), "lr": lr * LLRD ** 4}]
    g += [{"params": list(st.parameters()), "lr": lr * LLRD ** (3 - s)} for s, st in enumerate(bb.encoder.stages)]
    g += [{"params": list(bb.layernorm.parameters()), "lr": lr}, {"params": list(net.head.parameters()), "lr": lr * 10}]
    n_grouped = sum(p.numel() for gr in g for p in gr["params"]); assert n_grouped == sum(p.numel() for p in net.parameters())
    return g

class EMA:
    def __init__(self, model):
        self.m = copy.deepcopy(model).eval(); self.step = 0
        for p in self.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        self.step += 1; d = min(EMA_DECAY, (1 + self.step) / (10 + self.step))
        for e, p in zip(self.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

def class_weights(y):
    n = np.bincount(y, minlength=K).astype(float); w = np.sqrt(n.max() / np.maximum(n, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float, device=device)

def tta_views(x, mod):
    if mod == "D": return [torch.rot90(v, k, dims=(2, 3)) for v in (x, torch.flip(x, dims=[3])) for k in range(4)]
    return [x, torch.flip(x, dims=[3])]

@torch.no_grad()
def predict(model, df, mod, tta=True):
    dl = DataLoader(LesionImages(df, mod, transforms(mod, False)), batch_size=64, shuffle=False,
                    num_workers=0 if len(df) < 1000 else 4, pin_memory=True)
    model.eval(); probs, embs = [], []
    for x, _ in dl:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16):
            views = tta_views(x, mod) if tta else [x]
            e = torch.stack([model.embed(v) for v in views]).mean(0)
            p = model.head(e).float().softmax(1)
        probs.append(p.cpu().numpy()); embs.append(e.float().cpu().numpy())
    return np.concatenate(probs), np.concatenate(embs)

## Training loop (one model per modality × fold; resumable)

In [ ]:
LOG = []
def run_fold(mod, f):
    cfg = MOD_CFG[mod]; s0 = TRAIN_SEED * 100 + f
    torch.manual_seed(s0); np.random.seed(s0); random.seed(s0)
    trval, test = les[les["fold"] != f], les[les["fold"] == f]
    tr, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SPLIT_SEED)   # same as v1
    g = torch.Generator(); g.manual_seed(s0)
    dl = DataLoader(LesionImages(tr, mod, transforms(mod, True)), batch_size=cfg["batch"], shuffle=True, generator=g,
                    num_workers=4, pin_memory=True, drop_last=True, persistent_workers=True)
    model = Net(cfg["backbone"]).to(device).to(memory_format=torch.channels_last)
    groups = param_groups(model, cfg["lr"])
    opt = torch.optim.AdamW(groups, weight_decay=WD)
    steps = cfg["epochs"] * math.ceil(len(dl) / cfg["accum"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[gr["lr"] for gr in groups], total_steps=steps, pct_start=0.1)
    crit = nn.CrossEntropyLoss(weight=class_weights(tr["y"].values), label_smoothing=LABEL_SMOOTH)
    scaler = torch.amp.GradScaler("cuda"); ema = EMA(model)
    best, ckpt, t_start = -1, f"{OUT}/net_{mod}_fold{f}.pt", time.time()
    torch.cuda.reset_peak_memory_stats()
    for ep in range(cfg["epochs"]):
        model.train(); t0, tot = time.time(), 0.0
        for it, (x, yb) in enumerate(dl):
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last); yb = yb.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(model(x), yb) / cfg["accum"]
            scaler.scale(loss).backward(); tot += loss.item() * cfg["accum"]
            if (it + 1) % cfg["accum"] == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); ema.update(model)
        m_raw = evaluate(va["y"].values, predict(model, va, mod, tta=False)[0])
        m_ema = evaluate(va["y"].values, predict(ema.m, va, mod, tta=False)[0])
        LOG.append(dict(seed=TRAIN_SEED, mod=mod, fold=f, epoch=ep, loss=tot / len(dl), sec=time.time() - t0,
                        val_f1_raw=m_raw["macro_f1"], val_f1_ema=m_ema["macro_f1"], val_auc_ema=m_ema["macro_auc"]))
        print(f"{mod} f{f} ep{ep:02d} loss {tot/len(dl):.3f} | val macroF1 raw {m_raw['macro_f1']:.4f} "
              f"ema {m_ema['macro_f1']:.4f} | {time.time()-t0:.0f}s", flush=True)
        if m_ema["macro_f1"] > best: best = m_ema["macro_f1"]; torch.save(ema.m.state_dict(), ckpt)
    pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    Pt, _ = predict(model, test, mod)
    pd.DataFrame(Pt, columns=CLASSES).assign(lesion_id=test["lesion_id"].values, y=test["y"].values, fold=f)\
      .to_csv(f"{OUT}/oof_{mod}_fold{f}.csv", index=False)
    _, E = predict(model, les, mod); np.save(f"{OUT}/emb_{mod}_fold{f}.npy", E.astype(np.float16))
    cost = dict(seed=TRAIN_SEED, mod=mod, fold=f, train_min=(time.time() - t_start) / 60,
                peak_gpu_gb=torch.cuda.max_memory_allocated() / 1e9, best_val_f1_ema=best)
    pd.DataFrame([cost]).to_csv(f"{OUT}/cost_{mod}_fold{f}.csv", index=False); print(cost)
    del model, ema, opt, dl; torch.cuda.empty_cache()

for mod in MODS_TO_RUN:
    for f in FOLDS_TO_RUN:
        if os.path.exists(f"{OUT}/emb_{mod}_fold{f}.npy"): print("skip", mod, f); continue
        run_fold(mod, f)

## Results: Tier-1 v2, per-class recall, v1 comparison, training cost

In [ ]:
def fold_table(oof, name):
    t = pd.DataFrame([dict(fold=f, **evaluate(d["y"].values, d[CLASSES].values)) for f, d in oof.groupby("fold")])
    print(f"\n== {name} ==\n", t.round(4).to_string(index=False))
    return t
tables, recall = {}, {}
for mod in ["C", "D"]:
    parts = [f"{OUT}/oof_{mod}_fold{f}.csv" for f in range(5)]
    if all(map(os.path.exists, parts)):
        oof = pd.concat(map(pd.read_csv, parts)); oof.to_csv(f"{OUT}/oof_{mod}.csv", index=False)
        tables[mod] = fold_table(oof, f"{mod} v2 (seed {TRAIN_SEED})")
        recall[mod] = recall_score(oof["y"], oof[CLASSES].values.argmax(1), labels=range(K), average=None, zero_division=0)
summary = pd.DataFrame({m: t.drop(columns="fold").agg(["mean", "std"]).T.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
                        for m, t in tables.items()}).T
summary.to_csv(f"{OUT}/tier1_v2_summary.csv"); summary

In [ ]:
rec = pd.DataFrame({f"{m} v2": r for m, r in recall.items()}, index=CLASSES)
v1_dir = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):     # optional: attach notebook-o2 (v1) to compare
    if os.path.basename(root) == "stage2" and "oof_D.csv" in files: v1_dir = root; break
if v1_dir:
    for m in ["C", "D"]:
        o = pd.read_csv(f"{v1_dir}/oof_{m}.csv")
        rec[f"{m} v1"] = recall_score(o["y"], o[CLASSES].values.argmax(1), labels=range(K), average=None, zero_division=0)
        print(m, "v1 pooled:", {k: round(v, 4) for k, v in evaluate(o["y"].values, o[CLASSES].values).items()})
rec.round(3)

In [ ]:
cost = pd.concat([pd.read_csv(f"{OUT}/{q}") for q in sorted(os.listdir(OUT)) if q.startswith("cost_")])
print("GPU minutes per model and peak memory:"); display(cost.groupby("mod")[["train_min", "peak_gpu_gb"]].agg(["mean", "max"]).round(2))
print("Total GPU hours this seed:", round(cost["train_min"].sum() / 60, 2))
log = pd.read_csv(f"{OUT}/training_log.csv")
print("Epoch of best EMA validation F1 (per model):")
log.loc[log.groupby(["mod", "fold"])["val_f1_ema"].idxmax(), ["mod", "fold", "epoch", "val_f1_ema"]]